In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install lifelines -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 17.9 MB/s eta 0:00:00


Imports

In [3]:
import os
import pandas as pd

from sklearn.feature_selection import VarianceThreshold
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

Define paths and constants

In [4]:
MAIN_TRAIN_DATA_PATH = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/hcc_train_full.csv"
MAIN_TEST_DATA_PATH = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/hcc_test_full.csv"

TRAIN_PATH = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/UnivariateCox/hcc_UniCox_train_selected.csv"
TEST_PATH = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/UnivariateCox/hcc_UniCox_test_selected.csv"
SELECTED_FEATURES_PATH = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/UnivariateCox/hcc_UniCox_selected_features.csv"

In [5]:
import os

os.makedirs("/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/UnivariateCox/", exist_ok=True)

In [6]:
train_df = pd.read_csv(MAIN_TRAIN_DATA_PATH)
test_df = pd.read_csv(MAIN_TEST_DATA_PATH)

print("Train data shape:", train_df.shape)
print("Test data shape:", test_df.shape)

train_df.head()

Train data shape: (1085, 1196)
Test data shape: (272, 1196)


,PATIENT_ID,SAMPLE_ID,SAMPLE_COVERAGE,TUMOR_PURITY,MSI_SCORE,TUMOR_SIZE,TMB_NONSYNONYMOUS,DIAGNOSIS_AGE,BMI,BILIARY_SCORE,...,NEOADJUVANT_SYSTEMIC_THERAPY_No,NEOADJUVANT_SYSTEMIC_THERAPY_Yes,ADJUVANT_SYSTEMIC_THERAPY_,ADJUVANT_SYSTEMIC_THERAPY_No,ADJUVANT_SYSTEMIC_THERAPY_Yes,PALLIATIVE_SYSTEMIC_THERPAY_,PALLIATIVE_SYSTEMIC_THERPAY_No,PALLIATIVE_SYSTEMIC_THERPAY_Yes,OS_MONTHS,OS_EVENT
0,P-0011104,P-0011104-T01-IM5,886.0,60.0,0.00,6.4,5.872318,53.0,19.2,0.716,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,82.296,0
1,P-0001336,P-0001336-T01-IM3,755.0,30.0,0.16,6.4,4.436621,61.0,40.7,0.763,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,51.324,1
2,P-0064295,P-0064295-T01-IM7,635.0,70.0,0.50,5.4,4.101736,74.0,24.4,0.532,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,6.600,1
3,P-0049360,P-0049360-T01-IM6,537.0,30.0,0.12,6.4,4.323490,61.0,37.2,0.737,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,24.480,0
4,P-0010532,P-0010532-T01-IM5,1056.0,30.0,0.16,3.8,2.936159,63.0,20.9,0.763,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,14.592,1


In [7]:
ID_COLUMNS = ["PATIENT_ID", "SAMPLE_ID"]
DURATION_COL = "OS_MONTHS"
EVENT_COL = "OS_EVENT"
TARGET_COLUMNS = [DURATION_COL, EVENT_COL]

COMPONENTS = 50
MIN_VARIANCE = 0.001

print("ID columns:", ID_COLUMNS)
print("Target columns:", TARGET_COLUMNS)
print("Selected feature count:", COMPONENTS)
print("Minimum variance threshold:", MIN_VARIANCE)

ID columns: ['PATIENT_ID', 'SAMPLE_ID']
Target columns: ['OS_MONTHS', 'OS_EVENT']
Selected feature count: 50
Minimum variance threshold: 0.001


In [8]:
feature_columns = [
    col for col in train_df.columns
    if col not in ID_COLUMNS + TARGET_COLUMNS
]

print("Candidate feature count:", len(feature_columns))
feature_columns[:15]

Candidate feature count: 1192


['SAMPLE_COVERAGE',
 'TUMOR_PURITY',
 'MSI_SCORE',
 'TUMOR_SIZE',
 'TMB_NONSYNONYMOUS',
 'DIAGNOSIS_AGE',
 'BMI',
 'BILIARY_SCORE',
 'MUT_ABL1',
 'MUT_ACVR1',
 'MUT_AGO2',
 'MUT_AKT1',
 'MUT_AKT2',
 'MUT_AKT3',
 'MUT_ALB']

In [9]:
X_train_full = train_df[feature_columns]

variance_selector = VarianceThreshold(threshold=MIN_VARIANCE)
variance_selector.fit(X_train_full)

stable_feature_columns = X_train_full.columns[variance_selector.get_support()].tolist()

print("Stable feature count after variance filtering:", len(stable_feature_columns))
stable_feature_columns[:15]

Stable feature count after variance filtering: 938


['SAMPLE_COVERAGE',
 'TUMOR_PURITY',
 'MSI_SCORE',
 'TUMOR_SIZE',
 'TMB_NONSYNONYMOUS',
 'DIAGNOSIS_AGE',
 'BMI',
 'BILIARY_SCORE',
 'MUT_ABL1',
 'MUT_ACVR1',
 'MUT_AGO2',
 'MUT_AKT1',
 'MUT_AKT3',
 'MUT_ALB',
 'MUT_ALK']

In [10]:
cox_feature_rows = []
failed_features = []

for index, feature_name in enumerate(stable_feature_columns, start=1):
    feature_df = train_df[[feature_name, DURATION_COL, EVENT_COL]].copy()

    feature_df = feature_df.replace([float("inf"), float("-inf")], pd.NA).dropna()

    if feature_df.empty:
        failed_features.append((feature_name, "empty after dropna"))
        continue

    if feature_df[feature_name].nunique(dropna=False) <= 1:
        failed_features.append((feature_name, "constant feature"))
        continue

    try:
        cph = CoxPHFitter(penalizer=0.1)
        cph.fit(
            feature_df,
            duration_col=DURATION_COL,
            event_col=EVENT_COL,
            show_progress=False
        )

        feature_predictions = cph.predict_partial_hazard(feature_df)
        feature_c_index = concordance_index(
            feature_df[DURATION_COL],
            -feature_predictions,
            feature_df[EVENT_COL]
        )

        cox_feature_rows.append({
            "feature": feature_name,
            "coef": float(cph.params_[feature_name]),
            "hazard_ratio": float(cph.hazard_ratios_[feature_name]),
            "p_value": float(cph.summary.loc[feature_name, "p"]),
            "univariate_c_index": float(feature_c_index),
            "c_index_strength": abs(float(feature_c_index) - 0.5)
        })

    except Exception as e:
        failed_features.append((feature_name, str(e)))

    if index % 200 == 0:
        print("Processed features:", index)
        print("Successful fits so far:", len(cox_feature_rows))

Processed features: 200
Successful fits so far: 200
Processed features: 400
Successful fits so far: 400
Processed features: 600
Successful fits so far: 600
Processed features: 800
Successful fits so far: 800


In [11]:
from statsmodels.stats.multitest import multipletests

cox_feature_summary = pd.DataFrame(cox_feature_rows)

if cox_feature_summary.empty:
    print("No successful univariate Cox fits were produced.")
    print("First 10 failed features:")
    print(failed_features[:10])

else:

    # -------------------------------------------------------
    # Benjamini-Hochberg False Discovery Rate (FDR)
    # -------------------------------------------------------

    cox_feature_summary["adj_p_value"] = multipletests(
        cox_feature_summary["p_value"],
        method="fdr_bh"
    )[1]

    # -------------------------------------------------------
    # Keep statistically significant genes
    # -------------------------------------------------------

    SIGNIFICANCE_LEVEL = 0.05

    significant_features = cox_feature_summary[
        cox_feature_summary["adj_p_value"] < SIGNIFICANCE_LEVEL
    ].copy()

    # -------------------------------------------------------
    # If FDR is too strict, fall back to raw p-value
    # -------------------------------------------------------

    if len(significant_features) < COMPONENTS:

        print(
            f"Only {len(significant_features)} genes passed FDR correction."
        )
        print("Using raw p-value ranking instead.")

        ranked_features = cox_feature_summary.sort_values(
            by=["p_value", "c_index_strength"],
            ascending=[True, False]
        )

    else:

        ranked_features = significant_features.sort_values(
            by=["adj_p_value", "c_index_strength"],
            ascending=[True, False]
        )

    cox_feature_summary = ranked_features.reset_index(drop=True)

    print("Successful Cox fits :", len(cox_feature_rows))
    print("Genes retained :", len(cox_feature_summary))

    display(cox_feature_summary.head(20))

Only 48 genes passed FDR correction.
Using raw p-value ranking instead.
Successful Cox fits : 938
Genes retained : 938


,feature,coef,hazard_ratio,p_value,univariate_c_index,c_index_strength,adj_p_value
0,METASTASIS_Yes,0.949633,2.584760,4.061045e-33,0.635530,0.135530,3.809260e-30
1,CURATIVE_INTENT_SURGERY_Yes,-0.920306,0.398397,3.671217e-32,0.653573,0.153573,1.536703e-29
2,CURATIVE_INTENT_SURGERY_No,0.917305,2.502537,4.914828e-32,0.653632,0.153632,1.536703e-29
3,METASTASIS_No,-0.890243,0.410556,7.176907e-30,0.631440,0.131440,1.682985e-27
4,MAJOR_LIVER_RESECTION_,0.876069,2.401442,3.100262e-29,0.647558,0.147558,5.816091e-27
5,RESECTION_MARGIN_,0.875033,2.398953,1.422897e-28,0.644216,0.144216,2.224462e-26
6,RESECTION_MARGIN_R0,-0.856480,0.424654,1.418610e-24,0.632016,0.132016,1.900937e-22
7,C_N_Yes,0.790716,2.204974,3.181065e-24,0.609613,0.109613,3.729799e-22
8,PNI_,0.765181,2.149384,9.444636e-23,0.628779,0.128779,9.843409e-21
9,LVI_,0.742769,2.101746,4.005329e-22,0.630813,0.130813,3.756998e-20


In [12]:
selected_features_df = cox_feature_summary.head(COMPONENTS).copy()
selected_features = selected_features_df["feature"].tolist()

print("Selected feature count:", len(selected_features))
selected_features[:20]

Selected feature count: 50


['METASTASIS_Yes',
 'CURATIVE_INTENT_SURGERY_Yes',
 'CURATIVE_INTENT_SURGERY_No',
 'METASTASIS_No',
 'MAJOR_LIVER_RESECTION_',
 'RESECTION_MARGIN_',
 'RESECTION_MARGIN_R0',
 'C_N_Yes',
 'PNI_',
 'LVI_',
 'C_N_No',
 'PALLIATIVE_SYSTEMIC_THERPAY_Yes',
 'PNI_No',
 'PALLIATIVE_SYSTEMIC_THERPAY_No',
 'MAJOR_LIVER_RESECTION_No',
 'MULTIFOCAL_DISEASE_Yes',
 'MULTIFOCAL_DISEASE_No',
 'LVI_No',
 'P_N_Negative',
 'ADJUVANT_SYSTEMIC_THERAPY_No']

In [13]:
# Build final train and test datasets without identifiers.
# PATIENT_ID and SAMPLE_ID are row identifiers, not predictive features.
# Keeping them in the selected modeling files can introduce patient-level bias.
train_df_final = train_df[selected_features + TARGET_COLUMNS].copy().reset_index(drop=True)
test_df_final = test_df[selected_features + TARGET_COLUMNS].copy().reset_index(drop=True)

assert not any(col in train_df_final.columns for col in ID_COLUMNS)
assert not any(col in test_df_final.columns for col in ID_COLUMNS)

print("Final train shape:", train_df_final.shape)
print("Final test shape:", test_df_final.shape)

train_df_final.head()

Final train shape: (1085, 52)
Final test shape: (272, 52)


,METASTASIS_Yes,CURATIVE_INTENT_SURGERY_Yes,CURATIVE_INTENT_SURGERY_No,METASTASIS_No,MAJOR_LIVER_RESECTION_,RESECTION_MARGIN_,RESECTION_MARGIN_R0,C_N_Yes,PNI_,LVI_,...,TUMOR_STAGE_Poorly differentiated,TUMOR_STAGE_Well differentiated,MUT_SHQ1,CNA_XPO1,CNA_REL,MUT_KRAS,NEOADJUVANT_SYSTEMIC_THERAPY_No,MUT_FGFR4,OS_MONTHS,OS_EVENT
0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,82.296,0
1,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,51.324,1
2,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,6.600,1
3,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,24.480,0
4,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,14.592,1


In [14]:
print("Missing values in final train data:", int(train_df_final.isna().sum().sum()))
print("Missing values in final test data:", int(test_df_final.isna().sum().sum()))

Missing values in final train data: 0
Missing values in final test data: 0


In [15]:
selected_features_df.reset_index(drop=True, inplace=True)
selected_features_df.head(20)

,feature,coef,hazard_ratio,p_value,univariate_c_index,c_index_strength,adj_p_value
0,METASTASIS_Yes,0.949633,2.584760,4.061045e-33,0.635530,0.135530,3.809260e-30
1,CURATIVE_INTENT_SURGERY_Yes,-0.920306,0.398397,3.671217e-32,0.653573,0.153573,1.536703e-29
2,CURATIVE_INTENT_SURGERY_No,0.917305,2.502537,4.914828e-32,0.653632,0.153632,1.536703e-29
3,METASTASIS_No,-0.890243,0.410556,7.176907e-30,0.631440,0.131440,1.682985e-27
4,MAJOR_LIVER_RESECTION_,0.876069,2.401442,3.100262e-29,0.647558,0.147558,5.816091e-27
5,RESECTION_MARGIN_,0.875033,2.398953,1.422897e-28,0.644216,0.144216,2.224462e-26
6,RESECTION_MARGIN_R0,-0.856480,0.424654,1.418610e-24,0.632016,0.132016,1.900937e-22
7,C_N_Yes,0.790716,2.204974,3.181065e-24,0.609613,0.109613,3.729799e-22
8,PNI_,0.765181,2.149384,9.444636e-23,0.628779,0.128779,9.843409e-21
9,LVI_,0.742769,2.101746,4.005329e-22,0.630813,0.130813,3.756998e-20


In [16]:
train_df_final.to_csv(TRAIN_PATH, index=False)
test_df_final.to_csv(TEST_PATH, index=False)
selected_features_df.to_csv(SELECTED_FEATURES_PATH, index=False)

print("Saved train file to:", TRAIN_PATH)
print("Saved test file to:", TEST_PATH)
print("Saved selected features file to:", SELECTED_FEATURES_PATH)

Saved train file to: /content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/UnivariateCox/hcc_UniCox_train_selected.csv
Saved test file to: /content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/UnivariateCox/hcc_UniCox_test_selected.csv
Saved selected features file to: /content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/UnivariateCox/hcc_UniCox_selected_features.csv


In [17]:
train_check = pd.read_csv(TRAIN_PATH)
test_check = pd.read_csv(TEST_PATH)
features_check = pd.read_csv(SELECTED_FEATURES_PATH)

train_id_columns_present = [col for col in ID_COLUMNS if col in train_check.columns]
test_id_columns_present = [col for col in ID_COLUMNS if col in test_check.columns]

print("Reloaded train shape:", train_check.shape)
print("Reloaded test shape:", test_check.shape)
print("Reloaded selected features shape:", features_check.shape)
print("ID columns present in reloaded train file:", train_id_columns_present)
print("ID columns present in reloaded test file:", test_id_columns_present)

assert not train_id_columns_present
assert not test_id_columns_present

features_check.head(10)

Reloaded train shape: (1085, 52)
Reloaded test shape: (272, 52)
Reloaded selected features shape: (50, 7)
ID columns present in reloaded train file: []
ID columns present in reloaded test file: []


,feature,coef,hazard_ratio,p_value,univariate_c_index,c_index_strength,adj_p_value
0,METASTASIS_Yes,0.949633,2.584760,4.061045e-33,0.635530,0.135530,3.809260e-30
1,CURATIVE_INTENT_SURGERY_Yes,-0.920306,0.398397,3.671217e-32,0.653573,0.153573,1.536703e-29
2,CURATIVE_INTENT_SURGERY_No,0.917305,2.502537,4.914828e-32,0.653632,0.153632,1.536703e-29
3,METASTASIS_No,-0.890243,0.410556,7.176907e-30,0.631440,0.131440,1.682985e-27
4,MAJOR_LIVER_RESECTION_,0.876069,2.401442,3.100262e-29,0.647558,0.147558,5.816091e-27
5,RESECTION_MARGIN_,0.875033,2.398953,1.422897e-28,0.644216,0.144216,2.224462e-26
6,RESECTION_MARGIN_R0,-0.856480,0.424654,1.418610e-24,0.632016,0.132016,1.900937e-22
7,C_N_Yes,0.790716,2.204974,3.181065e-24,0.609613,0.109613,3.729799e-22
8,PNI_,0.765181,2.149384,9.444636e-23,0.628779,0.128779,9.843409e-21
9,LVI_,0.742769,2.101746,4.005329e-22,0.630813,0.130813,3.756998e-20


This output means the updated UnivariateCox feature-selection logic is working as intended.

The notebook fitted one separate Cox model per candidate feature using only the training data. For each feature, it recorded the Cox coefficient, hazard ratio, raw p-value, univariate C-index, C-index strength, and FDR-adjusted p-value. Then it ranked features primarily by `adj_p_value`, with `c_index_strength` used only as the secondary criterion.

In this run, enough features passed FDR correction, so the notebook did **not** need to fall back to raw p-value ranking. The selected 50 features are therefore the top FDR-significant survival-associated features, prioritized by statistical evidence first.

**How These Features Were Selected**
| Step | What Happened | Purpose |
|---|---|---|
| 1 | Each feature was tested independently using a univariate CoxPH model. | Measures whether that single feature is associated with survival. |
| 2 | Cox coefficient and hazard ratio were saved. | Shows direction and effect size. |
| 3 | Raw p-value was calculated. | Tests statistical association with survival. |
| 4 | Benjamini-Hochberg FDR correction was applied. | Controls false discoveries across many feature tests. |
| 5 | Features with `adj_p_value < 0.05` were retained. | Keeps statistically reliable survival-associated features. |
| 6 | Features were sorted by `adj_p_value`, then `c_index_strength`. | Ranks strongest statistical evidence first, then better discrimination. |
| 7 | Top 50 features were selected. | Final selected UnivariateCox feature set. |

**Interpretation of the Top Features**
| Feature | Direction | Hazard Ratio Meaning |
|---|---|---|
| `METASTASIS_Yes` | Risk-increasing | HR `2.58`, patients with metastasis have higher hazard/death risk. |
| `CURATIVE_INTENT_SURGERY_Yes` | Protective | HR `0.40`, associated with lower hazard. |
| `CURATIVE_INTENT_SURGERY_No` | Risk-increasing | HR `2.50`, absence of curative surgery is associated with higher hazard. |
| `METASTASIS_No` | Protective | HR `0.41`, absence of metastasis is associated with lower hazard. |
| `RESECTION_MARGIN_R0` | Protective | HR `0.42`, clean margin is associated with lower hazard. |
| `C_N_Yes` | Risk-increasing | HR `2.20`, clinically positive nodal status increases hazard. |
| `PALLIATIVE_SYSTEMIC_THERPAY_Yes` | Risk-increasing | HR `1.94`, likely reflects more advanced disease/treatment context. |

A positive coefficient means the feature increases predicted hazard. A hazard ratio greater than `1` means higher death risk. A negative coefficient means the feature is protective. A hazard ratio below `1` means lower death risk.

Important observation: pairs like `METASTASIS_Yes` and `METASTASIS_No`, or `CURATIVE_INTENT_SURGERY_Yes` and `CURATIVE_INTENT_SURGERY_No`, both appear because one-hot encoded categories are being evaluated independently. This is expected in univariate screening, but in multivariable CoxPH training, penalization helps manage redundancy.

Conclusion: the new method selected features based on stronger statistical survival association rather than just individual predictive discrimination. This makes the UnivariateCox feature set more defensible for research because it combines survival significance, effect direction, hazard ratio interpretation, and C-index support.